# Analyzing IT Job Market Trends: A LinkedIn Job Listing Scraping Project

### Project Overview
This project aims to analyze LinkedIn job postings to identify the most in-demand skills within the IT industry. By leveraging web scraping techniques, we extract job descriptions, process them to detect frequently mentioned programming languages, tools, and frameworks, and visualize emerging trends in the job market. This analysis provides valuable insights for job seekers, recruiters, and industry professionals, helping them align their skill sets with market demands.

The code provided demonstrates how to retrieve job vacancies using defined search criteria, parse job descriptions, and count the frequency of key skills.
We will also visualize the results in a bar chart to showcase the distribution of skills, providing actionable insights for both job seekers and recruiters.


#### Importing Required Libraries

To facilitate data extraction and processing, we utilize key Python libraries, including pandas for data handling, BeautifulSoup for parsing HTML, requests for sending HTTP requests, and time for managing request intervals to prevent rate limits.



In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
import requests
import time
import urllib.parse
import uuid

#### Defining Keywords for Job Search
Users can define specific job-related keywords to filter LinkedIn job listings based on their interests. This step allows customization of job searches by targeting specific roles, technologies, or industries.

In [ ]:
user_answer = "yes"
keywords = []
if user_answer == input("do want to choose vacancy manually?: "):
    while user_answer == "yes":
       keywords.append(input("enter keywords: "))
       user_answer = input("enter yes or no: ")
else:
    keywords = [
    "Data Scientist",
    "Software Developer",
    "Cybersecurity Engineer",
    "Cloud Engineer",
    "AI/Machine Learning Specialist",
    "Full Stack Developer",
    "DevOps Engineer",
    "Systems Administrator",
    "Database Administrator",
    "Mobile App Developer",
    "Network Engineer",
    "Business Intelligence Analyst",
    "Product Manager (Tech)",
    "IT Support Specialist",
    "Blockchain Developer",
    "Web Developer",
    "QA Engineer",
    "Game Developer",
    "UX/UI Designer",
    "Technical Project Manager",
    "Cloud Architect",
    "Embedded Systems Engineer",
    "Penetration Tester",
    "Big Data Engineer",
    "Site Reliability Engineer"
]

max_vacancy = int(input("How many vacancies do you want? "))

#### Fetching Job Listings
This section configures the query parameters and target URLs required to extract job postings from LinkedIn. The script sends requests to retrieve job postings that match predefined keywords and locations.

In [ ]:
l = []
target_url='https://www.linkedin.com/jobs-guest/jobs/api/seeMoreJobPostings/search?{}&location=Tel%20Aviv&geoId=101620260&start={}'
for keyword in keywords:
    for i in range(0, max_vacancy, 10):
        time.sleep(2)
        res = requests.get(target_url.format(urllib.parse.urlencode({"keywords": keyword }),i))
        soup = BeautifulSoup(res.text,'html.parser')
        alljobs_on_this_page = soup.find_all("li")
        for x in range(0,len(alljobs_on_this_page)):
            try:
                jobid = alljobs_on_this_page[x].find("div",{"class":"base-card"}).get('data-entity-urn').split(":")[3]
                l.append(jobid)
            except:
                time.sleep(2)
                try:
                    jobid = alljobs_on_this_page[x].find("div", {"class": "base-card"}).get('data-entity-urn').split(":")[3]
                    l.append(jobid)
                except:
                    print("vacancy not found", keyword)

#### Extracting Job IDs from HTML
Using BeautifulSoup, we parse the LinkedIn job listing HTML response to extract unique job identifiers (Job IDs). These IDs serve as reference points for further scraping and detailed analysis.

In [ ]:
jobs_table = {"vacancy_title":[],"company_title":[],"description":[],"seniority_level":[],"employment_type":[],"job_function":[],"industries":[]}
l = list(set(l))
for id in l:
    time.sleep(2)
    print(f"{l.index(id) + 1} / {len(l)}")
    job_link = "https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/" + id
    res = requests.get(job_link)
    soup = BeautifulSoup(res.text,'html.parser')
    try:
        vacancy_title = soup.find("h2",{"class":"top-card-layout__title"}).text
    except:
        vacancy_title = ""
    try:
        company_title = soup.find("a",{"class":"topcard__org-name-link"}).text.strip()
    except:
        company_title = ""
    try:
        job_description = soup.find("div", {"class": "show-more-less-html__markup"}).text.strip()
    except:
        job_description = ""

#### Exporting Results to Excel
To facilitate further analysis and reporting, we export the extracted job data (including skill counts and job descriptions) into an Excel file. This allows users to review, filter, and refine the dataset efficiently.

In [ ]:
    try:
        job_criteria = soup.find_all("span", {"class": "description__job-criteria-text"})
        seniority_level = job_criteria[0].text.strip()
        employment_type = job_criteria[1].text.strip()
        job_function = job_criteria[2].text.strip()
        industries = job_criteria[3].text.strip()
    except:
        seniority_level = ""
        employment_type = ""
        job_function = ""
        industries = ""
    jobs_table["vacancy_title"].append(vacancy_title)
    jobs_table["company_title"].append(company_title)
    jobs_table["description"].append(job_description)
    jobs_table["seniority_level"].append(seniority_level)
    jobs_table["employment_type"].append(employment_type)
    jobs_table["job_function"].append(job_function)
    jobs_table["industries"].append(industries)

df = pd.DataFrame(jobs_table)
df.to_excel(f"{uuid.uuid4()}.xlsx")

#### Keyword Analysis for Various Fields
Several keyword-specific datasets have been uploaded, containing job postings related to specific skills such as Python, Machine Learning, SQL, UX, and Tableau. These datasets allow us to analyze how frequently each skill appears in job descriptions across different roles.

#### Concatenate the Data
To streamline the analysis, we merge multiple job listings datasets into a single consolidated dataframe using pandas.concat(). This combined dataset allows for a comprehensive analysis of job vacancies across different keywords. The final dataset is exported as ALL_VACANCY.xlsx.

In [8]:
import os
import pandas as pd


files_list = []
ignore_files = ["all_vacancy.xlsx","skills.xlsx", "skills1.xlsx", "test.xlsx"]
for file in os.listdir():
    if file.endswith(".xlsx") and file not in ignore_files:
        files_list.append(pd.read_excel(file))

concat = pd.concat(files_list)
concat.to_excel("tes.xlsx", index=False)
print(concat)

    Unnamed: 0                   vacancy_title    company_title  \
0            0  Python Developer (Entry Level)    SynergisticIT   
1            1    Data Scientist - Real Estate         Circle K   
2            2                  Data Scientist    Ample Insight   
3            3            AI/ML Data Scientist  Acuity Insights   
4            4            Automation Developer    Martell Media   
..         ...                             ...              ...   
95          95                             NaN              NaN   
96          96                             NaN              NaN   
97          97                             NaN              NaN   
98          98                             NaN              NaN   
99          99                             NaN              NaN   

                                          description   seniority_level  \
0   The Job Market is Challenging due to almost 30...       Entry level   
1   JOIN OUR TEAM!At Couche-Tard/Circle K, ou

#### Skills Analysis
We extract in-demand skills from job descriptions and classify them into the following categories:
Technical Skills (e.g., Python, SQL, Cloud Computing)
Soft Skills (e.g., Communication, Teamwork)
Analytical Skills (e.g., Problem-Solving, Data Interpretation)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_excel('ALL_VACANCY.xlsx')
# Extract the 'description' column for analysis to identify technical skills
job_descriptions = df['description'].dropna()

# Common technical skills to search for in descriptions
linkedin_skills = [
    # Programming Languages
    "Python", "JavaScript", "Java", " C++ ", "C#", "Ruby", "PHP", "Swift",
    "TypeScript", " Go ", " R ", "Kotlin", "SQL", "Rust", "MATLAB",

    # Web Development
    "HTML", "CSS", "JavaScript", "React.js", "Angular", "Vue.js", "SASS/SCSS",
    "Bootstrap", "Tailwind CSS", "Node.js", "Django", "Flask", "ASP.NET",
    "Ruby on Rails", "Next.js", "WebSockets",

    # Mobile Development
    "React Native", "Flutter", "Swift (iOS)", "Kotlin (Android)",
    "Java (Android)", "Xamarin", "Ionic",

    # Database Management
    "MySQL", "PostgreSQL", "MongoDB", "SQLite", "Microsoft SQL Server",
    "Redis", "Cassandra", "Firebase Realtime Database", "DynamoDB",

    # Cloud and DevOps
    "AWS (Amazon Web Services)", "Microsoft Azure", "Google Cloud Platform (GCP)",
    "Docker", "Kubernetes", "Jenkins", "Terraform", "Ansible", "GitHub Actions",
    "CircleCI", "CloudFormation",

    # Data Analysis & Machine Learning
    "Pandas", "NumPy", "Matplotlib", "Scikit-learn", "TensorFlow", "PyTorch",
    "Keras", "Apache Spark", "Hadoop", "Tableau", "Power BI",

    # Version Control
    "Git", "GitHub", "GitLab", "Bitbucket",

    # Testing and Debugging
    "Selenium", "Cypress", "Jest", "Mocha", "JUnit", "Pytest",
    "Postman", "SoapUI", "Debugging Tools (Chrome DevTools, Xcode Debugger, etc.)",

    # Game Development
    "Unity", "Unreal Engine", "Godot", "CryEngine", "Cocos2d",

    # System Programming
    "Assembly", "Rust", " C ", "Kernel Development", "Embedded Systems",

    # Networking and Security
    "Linux Networking", "Wireshark", "OpenSSL", "Network Protocols (TCP/IP, HTTP/HTTPS)",
    "Penetration Testing Tools (Metasploit, Burp Suite)",
    "Firewalls and IDS/IPS", "Cryptography Libraries (PyCrypto, Bouncy Castle)",

    # Scripting
    "Shell Scripting (Bash, Zsh)", "PowerShell", "Perl", "TCL",

    # Artificial Intelligence and Data Science
    "Natural Language Processing (NLP)", "OpenCV (Computer Vision)",
    "Reinforcement Learning", "Time Series Analysis", "Deep Learning Frameworks",
    "AI Model Deployment (ONNX, TensorRT)",

    # Project Management Tools
    "JIRA", "Trello", "Asana", "Notion", "Monday.com",

    # API Development and Integration
    "RESTful APIs", "GraphQL", "gRPC", "Webhooks", "OAuth",

    # Others
    "WebAssembly", "Blockchain Development (Solidity, Ethereum, Hyperledger)",
    "IoT Development (Arduino, Raspberry Pi)",
    "ERP/CRM Systems (SAP, Salesforce)",

    # Technical Skills
    "Artificial Intelligence (AI) and Machine Learning",
    "Data Analysis and Interpretation",
    "Digital Literacy and Tech Proficiency",
    "Cybersecurity",
    "Cloud Computing",
    "Software Development",
    "Blockchain Technology",
    "Internet of Things (IoT)",
    "UX/UI Design",
    "Mobile Application Development",

    # Soft Skills
    "Critical Thinking and Problem-Solving",
    "Adaptability and Flexibility",
    "Emotional Intelligence",
    "Creativity and Innovation",
    "Communication and Collaboration",
    "Leadership and People Management",
    "Time Management",
    "Negotiation",
    "Decision-Making",
    "Stress Management",

    # Hybrid Skills
    "Digital Marketing",
    "Project Management",
    "Financial Management",
    "Sales and Business Development",
    "Strategic Planning",
    "Customer Service",
    "Content Creation",
    "Data Visualization",
    "SEO/SEM",
    "Supply Chain Management"
]

# Count the occurrence of each skill in the job descriptions
skill_counts = {skill: sum(job_descriptions.str.contains(skill, case=False)) for skill in linkedin_skills}

# Convert to a DataFrame for better readability and sort by frequency
skills_df = pd.DataFrame(list(skill_counts.items()), columns=['Skill', 'Count']).sort_values(by='Count', ascending=False)
skills_df = skills_df[skills_df['Count'] > 0]
skills_df.to_excel('summary_skills.xlsx')
plt.bar(skills_df["Skill"], skills_df["Count"], color='violet')
plt.show()

## Part 2 - Research Description 
The next phase of the research will focus on comparing the popularity of IT vacancies across four regions using LinkedIn as the primary data source. The selected regions are Canada, the USA, Israel, and the European Union (EU). These regions were chosen as central areas of focus due to their significant presence and influence in the global tech industry, sharing similarities in their technological advancements and demand for skilled IT professionals.

We used ChatGPT Plus to generate an initial hypothesis regarding the most popular IT vacancies in each region. Based on the insights provided by ChatGPT, the top IT vacancies in each region are as follows:

#### Canada:
1. Senior Software Engineer (Source: Robertson College)
2. IT Project Manager (Source: Robertson College)
3. User Experience (UX) Designer (Source: Robertson College)
4. DevOps Engineer (Source: Robertson College)
5. Back-End Developer (Source: Robertson College)

#### USA:
1. Software Developer (Source: Indeed)
2. Information Security Analyst (Source: U.S. News)
3. Data Scientist (Source: Forbes)
4. Cloud Engineer (Source: Forbes)
5. IT Support Specialist (Source: U.S. News)

#### Israel:
1. Software Engineer (Source: Tech Career Israel)
2. Cybersecurity Specialist (Source: Tech Career Israel)
3. Data Analyst (Source: Startup Jobs Israel)
4. Full-Stack Developer (Source: Tech Career Israel)
5. Cloud Solutions Architect (Source: Startup Jobs Israel)

#### European Union (EU): 
1. Software Developer (Source: Europass)
2. IT Project Manager (Source: EuroJobs)
3. Data Scientist (Source: LinkedIn)
4. Network Engineer (Source: EuroJobs)
5. Cloud Engineer (Source: LinkedIn)

Upon reviewing the sources cited for each region, we observed a variation in their reliability and prominence. To ensure consistency and comparability, we decided to use LinkedIn as the primary source for data collection and analysis. LinkedIn consistently ranked among the top job search platforms across all four regions, as highlighted by ChatGPT, making it a credible and widely recognized platform for analyzing IT job trends.
This approach allows us to validate and refine the information provided by ChatGPT while ensuring the data is gathered from a uniform and reliable source.

#### Research Hypotheses 
##### Hypothesis 1:
 IT job vacancies on LinkedIn will show significant regional variations in popularity compared to AI-generated predictions.
##### Hypothesis 2: 
The most frequently required skills in IT job postings on LinkedIn will differ from those predicted by ChatGPT.
##### Hypothesis 3:
While skill requirements vary, certain foundational IT skills will be common across all regions. 


The script sets up a dictionary of locations (locations) and a list of IT job roles (keywords).
The keywords list contains various job titles like Data Scientist, Cybersecurity, Full Stack Developer, DevOps Engineer, etc.

Scraping Job Listings: The function get_count_of_jobs(keywords, location) constructs LinkedIn job search URLs dynamically using the urllib.pars .urlencode() function. It sends HTTP requests to retrieve job postings from LinkedIn.
Using BeautifulSoup, it parses the HTML response and extracts the total job count for a given keyword and region.

Storing the Results:
The script iterates over all job titles and locations, appends the extracted job counts into a summary dictionary (summary_table), and exports the data to Excel (summary_table.xlsx).  

In [ ]:
import time 
import urllib
import urllib.parse
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [ ]:
locations = {"canada": 101174742, "israel": 101620260, "usa": 103644278, "eu": 91000000}
keywords = [
    "Data Scientist",
    "Software Developer",
    "Cybersecurity",
    "Cloud Engineer",
    "AI/Machine Learning Specialist",
    "Full Stack Developer",
    "DevOps Engineer",
    "Systems Administrator",
    "Database Administrator",
    "Mobile App Developer",
    "Network Engineer",
    "Business Intelligence Analyst",
    "Product Manager (Tech)",
    "IT Support Specialist",
    "Blockchain Developer",
    "Web Developer",
    "QA Engineer",
    "Game Developer",
    "UX/UI Designer",
    "Technical Project Manager",
    "Cloud Architect",
    "Embedded Systems Engineer",
    "Penetration Tester",
    "Big Data Engineer",
    "Site Reliability Engineer"
]


def get_count_of_jobs(keywords, location):
    base_url = "https://www.linkedin.com/jobs/search?"
    query = "{}&geoId={}&trk=public_jobs_jobs-search-bar_search-submit&position=1&pageNum=0".format(urllib.parse.urlencode({"keywords": keywords}), location)
    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}
    print(base_url + query)
    response = requests.get(base_url + query, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    quantity = 0
    try:
        quantity = int(soup.find('span', class_='results-context-header__job-count')
                       .text
                       .replace(',', '')
                       .replace('+', ''))
    except:
        print("r")
    time.sleep(2)
    return quantity

summary_table = {"location":[],"job_title":[],"quantity":[]}
for location in locations:
    for keyword in keywords:
        summary_table["location"].append(location)
        summary_table["job_title"].append(keyword)
        summary_table["quantity"].append(get_count_of_jobs(keyword, locations[location]))

df = pd.DataFrame(summary_table)
df.to_excel("summary_table.xlsx")

### Hypothesis 1
#### Visualization of Vacancy Popularity Results
IT job vacancies on LinkedIn will show significant regional variations in popularity compared to AI-generated predictions.

#### Key Features of the Code:

##### Loading Job Vacancy Data:
Reads the summary_table.xlsx file containing job vacancy counts per region.

##### Filtering Data for Visualization:
The script filters the top 10 most popular job titles per region.

##### Generating Pie Charts:
Uses Matplotlib to create pie charts for each region.
Labels each section with the job role name and its percentage share.

Saves the generated images as canada.png, usa.png, israel.png, eu.png.

  

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


df = pd.read_excel('summary_table.xlsx')
countries = ["canada", "israel", "usa", "eu"]

for country in countries:
    df_filter = df[df["location"] == country].sort_values(by=["quantity"], ascending=False).head(10)
    plt.pie(df_filter["quantity"], labels=df_filter["job_title"], autopct="%1.1f%%")
    plt.title(country)
    plt.savefig(f"{country}.png")
    plt.show()

<img src="usa.png" width=500>
<img src="eu.png" width=500>
<img src="canada.png" width=500>
<img src="israel.png" width=500>

### Hypothesis 2 & 3 
#### Visualization of Skills Popularity Results 

Hypothesis 2: The most frequently required skills in IT job postings on LinkedIn will differ from those predicted by ChatGPT.

Hypothesis 3: While skill requirements vary, certain foundational IT skills will be common across all regions.

##### Key Features of the Code

##### Scraping Job Descriptions:
Iterates over a list of job IDs (id_vacancy_list) and sends requests to LinkedIn’s job posting pages.

##### Extracts:
Job Title
Full Job Description
Region
Stores the extracted data in a dictionary (jobs_table).

##### Saving Results:
Converts the dictionary into a Pandas DataFrame.
Exports the job descriptions as an Excel file for analysis.

##### Extracting Skills from Job Descriptions:
Reads description_country.xlsx, which contains job descriptions categorized by region.
Uses regular expressions (re module) to clean the text and extract relevant words.
Iterates over predefined IT skills per region and counts their occurrences.



In [ ]:
jobs_table = {"vacancy_title":[],"description":[],"region":[]}
id_vacancy_list = l
for id,region in id_vacancy_list :
    time.sleep(2)
    job_link = "https://www.linkedin.com/jobs-guest/jobs/api/jobPosting/" + id
    res = requests.get(job_link)
    soup = BeautifulSoup(res.text,'html.parser')
    try:
        vacancy_title = soup.find("h2",{"class":"top-card-layout__title"}).text
    except:
        vacancy_title = ""
    try:
        job_description = soup.find("div", {"class": "show-more-less-html__markup"}).text.strip()
    except:
        job_description = ""
    jobs_table["vacancy_title"].append(vacancy_title)
    jobs_table["description"].append(job_description)
    jobs_table["region"].append(region)

df = pd.DataFrame(jobs_table)
df.to_excel(f"{uuid.uuid4()}.xlsx")

#### Canada:
Cloud Computing: AWS, Microsoft Azure, and Google Cloud.
Data Analysis & Visualization: SQL, Python, R, Tableau, and Power BI.
Artificial Intelligence (AI) & Machine Learning: neural networks, NLP, TensorFlow, and PyTorch.
Software Development:  Python, Java, and C++.
Agile Project Management:  Scrum, Kanban, and using tools like Jira or Trello.
#### USA
Artificial Intelligence & Machine Learning: Skills in data modeling, predictive analytics, and frameworks like Scikit-learn or Keras.
Data Science:  Python, R, SQL, and advanced statistics.
Cybersecurity: Skills in risk management, network security, penetration testing, and tools like Splunk or Wireshark.
Cloud Engineering: Experience in Kubernetes, Docker, and cloud platforms (AWS, Azure, Google Cloud).
Full-Stack Development: Knowledge of React, Angular, Node.js, and APIs.
#### Israel
Cybersecurity: Mastery in ethical hacking, cryptography, and SIEM tools like QRadar.
Data Science: Proficiency in data mining, machine learning models, and platforms like Apache Spark.
Software Engineering: Strong knowledge in Java, C#, and DevOps practices.
Artificial Intelligence: Skills in developing AI algorithms, NLP, and robotics.
Cloud Infrastructure: Expertise in multi-cloud environments and automation tools like Ansible or Terraform.
#### European Union (EU)
Information and Communication Technology (ICT): Broad IT knowledge in networking, hardware, and system integration.
Artificial Intelligence: Skills in developing AI models, big data processing, and computer vision.
Data Science: Advanced analytics using Python, SAS, and SQL, as well as knowledge of GDPR compliance.
Cybersecurity: Proficiency in EU-specific security protocols and risk assessment frameworks.
Cloud Engineering: Experience with hybrid cloud solutions and edge computing.

In [ ]:
import re
import pandas as pd
from matplotlib import pyplot as plt


def remove_punctuation(text):
    return re.sub(r'[^\w\s]', '', text)


popular_it_skills_by_region = {
    "Canada": [
        "Cloud Computing (AWS, Azure, Google Cloud)",
        "Data Analysis & Visualization (SQL, Python, R, Tableau, Power BI)",
        "Artificial Intelligence & Machine Learning (TensorFlow, PyTorch)",
        "Software Development (Python, Java, C++)",
        "Agile Project Management (Scrum, Kanban, Jira, Trello)"
    ],
    "USA": [
        "Artificial Intelligence & Machine Learning (TensorFlow, PyTorch, Scikit-learn)",
        "Data Science (Python, R, SQL)",
        "Cybersecurity (risk management, network security, penetration testing)",
        "Cloud Engineering (Kubernetes, Docker, AWS, Azure)",
        "Full-Stack Development (React, Angular, Node.js)"
    ],
    "Israel": [
        "Cybersecurity (ethical hacking, cryptography, SIEM tools)",
        "Data Science (machine learning, Apache Spark)",
        "Software Engineering (Java, C#)",
        "Artificial Intelligence (NLP, robotics)",
        "Cloud Infrastructure (multi-cloud management, Ansible, Terraform)"
    ],
    "EU": [
        "Information and Communication Technology (networking, hardware, system integration)",
        "Artificial Intelligence (big data processing, computer vision)",
        "Data Science (Python, SAS, SQL, GDPR-compliant analytics)",
        "Cybersecurity (EU-specific protocols, risk assessment)",
        "Cloud Engineering (hybrid cloud, edge computing)"
    ]
}

In [ ]:
skills_by_region = []
for region, skills in popular_it_skills_by_region.items():
    print(f"{region}:")
    for skill in skills:
        skills_by_region.extend((skill.split("(")[1].replace(")", "").split(", ")))
skills_by_region = list(set(skills_by_region))
df = pd.read_excel('description_country.xlsx')
regions = df["region"].unique().tolist()
words_list = {}
word_count = {}

Initializes a dictionary for each region and iterates through the descriptions to clean the text and count the occurrences of predefined IT skills. After processing, it sorts these skills by frequency, extracts the top 10 skills for each region, and visualizes the results using pie charts, which are then saved as image files.

In [ ]:
for region in regions:
    words_list[region] = {}
    for description in df[df["region"] == region]["description"]:
        clear_description = remove_punctuation(description.lower())
        for skill in linkedin_skills:
            if skill.lower() in clear_description:
                if skill.lower() in words_list[region]:
                    words_list[region][skill.lower()] += 1
                else:
                    words_list[region][skill.lower()] = 1

for region in regions:
    words_list[region] = (sorted(words_list[region].items(), key=lambda item: item[1], reverse=True))
    skills = []
    values = []
    for element in words_list[region][:10]:
        skills.append(element[0])
        values.append(element[1])
    plt.pie(values, labels=skills, autopct='%1.1f%%')
    plt.title(region)
    plt.savefig(f"{region}_-final.png")
    plt.show()

<img src="canada_-final.png" width=500>
<img src="eu_-final.png" width=500>
<img src="israel_-final.png" width=500>
<img src="usa_-final.png" width=500>


### Conclusions - Comparison Between Hypotheses and ChatGPT-Generated Skills Data

##### Regional Variability:
 Job vacancies and skill demands significantly differ across regions. For example, Israel prioritizes Cybersecurity, while Canada emphasizes Data Visualization Tools.

##### AI vs. Real Data:
 ChatGPT provided reasonable insights, but real-world LinkedIn data revealed unexpected trends, such as higher-than-expected demand for Cloud Infrastructure roles.

##### Global Skill Demand:
 Core IT skills like Python, SQL, and Cloud Computing remain universally in demand, validating Hypothesis 3.